In [2]:
# ============================================================
# 病原菌計數統計（按月份 + 14天固定視窗去重複）
# 從 Google Drive 匯入，輸出到 Google Drive 指定資料夾
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ↓↓↓ 只需修改這一行 ↓↓↓
SEARCH_ROOT = '/content/drive/MyDrive/projects'
# ↑↑↑ 只需修改這一行 ↑↑↑

import pandas as pd
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# ════════════════════════════════════════════════════════════
# 工具函式
# ════════════════════════════════════════════════════════════
def list_csv_files(folder_path):
    if not os.path.exists(folder_path):
        return []
    return sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith('.csv')
    ])

def list_all_folders(base):
    if not os.path.exists(base):
        print(f"⚠️ 路徑不存在：{base}")
        return [base]
    folders = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        folders.append(root)
        if len(folders) > 200:
            break
    return sorted(folders)

all_folders = list_all_folders(base=SEARCH_ROOT)
print(f"✅ 找到 {len(all_folders)} 個資料夾（搜尋範圍：{SEARCH_ROOT}）")

# ════════════════════════════════════════════════════════════
# 步驟一：選擇輸入 CSV
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("【步驟 1】選擇病原菌細項 CSV 檔")
print("="*55)

folder_selector_in = widgets.Dropdown(
    options=all_folders, description='資料夾：',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '80px'}
)
file_selector_in = widgets.Select(
    options=[], description='CSV 檔：',
    layout=widgets.Layout(width='600px', height='120px'),
    style={'description_width': '80px'}
)

def on_folder_change_in(change):
    file_selector_in.options = list_csv_files(change['new'])

folder_selector_in.observe(on_folder_change_in, names='value')
file_selector_in.options = list_csv_files(all_folders[0])
display(folder_selector_in, file_selector_in)

# ════════════════════════════════════════════════════════════
# 步驟二：選擇輸出資料夾 + 輸入檔名
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("【步驟 2】選擇輸出資料夾與檔名")
print("="*55)

folder_selector_out = widgets.Dropdown(
    options=all_folders, description='輸出資料夾：',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '100px'}
)

filename_all = widgets.Text(
    value='pathogen_monthly_all',
    description='全部檢體檔名：',
    placeholder='不需加 .csv',
    layout=widgets.Layout(width='420px'),
    style={'description_width': '120px'}
)

filename_blood = widgets.Text(
    value='pathogen_monthly_blood',
    description='Blood檢體檔名：',
    placeholder='不需加 .csv',
    layout=widgets.Layout(width='420px'),
    style={'description_width': '120px'}
)

display(folder_selector_out, filename_all, filename_blood)

# ════════════════════════════════════════════════════════════
# 步驟三：執行按鈕
# ════════════════════════════════════════════════════════════
run_button = widgets.Button(
    description='▶ 執行統計並輸出',
    button_style='success',
    layout=widgets.Layout(width='220px', height='40px', margin='20px 0')
)
output_area = widgets.Output()
display(run_button, output_area)

# ════════════════════════════════════════════════════════════
# 14天固定視窗去重複函數
# ════════════════════════════════════════════════════════════
def deduplicate_14days(df_input):
    df_sorted = df_input.sort_values(['病歷號', '菌種', '年月', '簽收日期']).copy()
    keep_rows = []

    for (pid, pathogen, month), group in df_sorted.groupby(['病歷號', '菌種', '年月']):
        window_start = None
        for _, row in group.iterrows():
            if window_start is None or (row['簽收日期'] - window_start).days > 14:
                keep_rows.append(row)
                window_start = row['簽收日期']

    if keep_rows:
        return pd.DataFrame(keep_rows, columns=df_input.columns)
    else:
        return pd.DataFrame(columns=df_input.columns)

# ════════════════════════════════════════════════════════════
# 執行主程式
# ════════════════════════════════════════════════════════════
def on_run_clicked(b):
    with output_area:
        clear_output()

        if not file_selector_in.value:
            print("❌ 請先選擇 CSV 檔案")
            return

        # ── 讀取檔案 ──────────────────────────────────────────
        filepath = file_selector_in.value
        print(f"📂 讀取檔案：{os.path.basename(filepath)}")

        df_raw = pd.read_csv(filepath, encoding='utf-8-sig')

        # ── 資料清理 ──────────────────────────────────────────
        df = df_raw.copy()
        df.columns = ['_group', '護理站', '菌種', '檢體', '病歷號', '姓名', '申請序號', '簽收日期']

        df = df[df['病歷號'].notna()].copy()

        df['簽收日期'] = df['簽收日期'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
        df['簽收日期'] = pd.to_datetime(df['簽收日期'], format='%Y%m%d', errors='coerce')
        df = df[df['簽收日期'].notna()].copy()

        df['病歷號'] = df['病歷號'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        df['菌種']   = df['菌種'].str.strip()
        df['檢體']   = df['檢體'].fillna('').str.strip()
        df['年月']   = df['簽收日期'].dt.to_period('M')

        print(f"✅ 清理後筆數：{len(df)}")
        print(f"✅ 涵蓋月份：{sorted(df['年月'].unique().astype(str))}")

        # ── 版本一：全部檢體 ──────────────────────────────────
        df_dedup_all = deduplicate_14days(df)
        result_all = (df_dedup_all
                      .groupby(['年月', '菌種'])
                      .size()
                      .reset_index(name='株數'))
        result_all['年月'] = result_all['年月'].astype(str)
        result_all = result_all.sort_values(['年月', '株數'], ascending=[True, False])

        # ── 版本二：僅 Blood 檢體 ─────────────────────────────
        df_blood = df[df['檢體'].str.lower().str.startswith('blood', na=False)].copy()
        df_dedup_blood = deduplicate_14days(df_blood)

        if not df_dedup_blood.empty:
            result_blood = (df_dedup_blood
                            .groupby(['年月', '菌種'])
                            .size()
                            .reset_index(name='株數'))
            result_blood['年月'] = result_blood['年月'].astype(str)
            result_blood = result_blood.sort_values(['年月', '株數'], ascending=[True, False])
        else:
            result_blood = pd.DataFrame(columns=['年月', '菌種', '株數'])

        # ── 預覽結果 ──────────────────────────────────────────
        print("\n" + "="*45)
        print("【版本一】全部檢體統計（依月份）")
        print("="*45)
        print(result_all.to_string(index=False))

        print("\n" + "="*45)
        print("【版本二】僅 Blood 檢體統計（依月份）")
        print("="*45)
        if result_blood.empty:
            print("（無 Blood 檢體資料）")
        else:
            print(result_blood.to_string(index=False))

        # ── 輸出到 Google Drive ───────────────────────────────
        out_folder = folder_selector_out.value
        os.makedirs(out_folder, exist_ok=True)

        fname_all = filename_all.value.strip() or 'pathogen_monthly_all'
        fname_blood = filename_blood.value.strip() or 'pathogen_monthly_blood'
        if not fname_all.endswith('.csv'):   fname_all   += '.csv'
        if not fname_blood.endswith('.csv'): fname_blood += '.csv'

        path_all   = os.path.join(out_folder, fname_all)
        path_blood = os.path.join(out_folder, fname_blood)

        result_all.to_csv(path_all,   index=False, encoding='utf-8-sig')
        result_blood.to_csv(path_blood, index=False, encoding='utf-8-sig')

        print(f"\n✅ 已儲存至 Google Drive：")
        print(f"   - {path_all}")
        print(f"   - {path_blood}")

run_button.on_click(on_run_clicked)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 找到 8 個資料夾（搜尋範圍：/content/drive/MyDrive/projects）

【步驟 1】選擇病原菌細項 CSV 檔


Dropdown(description='資料夾：', layout=Layout(width='600px'), options=('/content/drive/MyDrive/projects', '/conte…

Select(description='CSV 檔：', layout=Layout(height='120px', width='600px'), options=(), style=DescriptionStyle(…


【步驟 2】選擇輸出資料夾與檔名


Dropdown(description='輸出資料夾：', layout=Layout(width='600px'), options=('/content/drive/MyDrive/projects', '/con…

Text(value='pathogen_monthly_all', description='全部檢體檔名：', layout=Layout(width='420px'), placeholder='不需加 .csv'…

Text(value='pathogen_monthly_blood', description='Blood檢體檔名：', layout=Layout(width='420px'), placeholder='不需加 …

Button(button_style='success', description='▶ 執行統計並輸出', layout=Layout(height='40px', margin='20px 0', width='2…

Output()

In [3]:
# ============================================================
# GitHub 版本控管設定
# 每次開新 Colab session 都需要執行一次
# ============================================================

import os
import subprocess

# ↓↓↓ 修改這三行 ↓↓↓
GITHUB_TOKEN    = 'GITHUB_TOKEN_REMOVED'
GITHUB_USERNAME = 'qq1690690'
GITHUB_REPO     = 'amr-analysis-project'
# ↑↑↑ 修改這三行 ↑↑↑

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
CLONE_DIR = f'/content/{GITHUB_REPO}'

# Git 基本設定
subprocess.run(['git', 'config', '--global', 'user.email', f'{GITHUB_USERNAME}@users.noreply.github.com'])
subprocess.run(['git', 'config', '--global', 'user.name', GITHUB_USERNAME])

# Clone repository 到 Colab
if os.path.exists(CLONE_DIR):
    print(f"✅ 資料夾已存在，執行 pull 更新...")
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'])
else:
    print(f"📂 Clone repository...")
    subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR])

print(f"\n✅ 完成！Repository 位置：{CLONE_DIR}")
print(f"目前檔案：")
for f in os.listdir(CLONE_DIR):
    print(f"   - {f}")

📂 Clone repository...

✅ 完成！Repository 位置：/content/amr-analysis-project
目前檔案：
   - .git
   - README.md
   - .gitignore
